In [2]:
import pandas as pd
import db_mgmt as mgmt
import os

In [3]:
os.getcwd()

'/Users/thiagorodrigues/Documents/CANOE/CANOE_ref'

In [314]:
data_schema = 'dbs/canoe_dataset_schema 4.sql'
db_file = 'dbs/canoe_transport.sqlite'
os.remove(db_file) if os.path.exists(db_file) else None
mgmt.convert_sql_to_sqlite(data_schema, db_file)
data = mgmt.sqlite_to_dfs(db_file)
data.keys()

dict_keys(['MetaData', 'MetaDataReal', 'SeasonLabel', 'SectorLabel', 'CapacityCredit', 'CapacityFactorProcess', 'CapacityFactorTech', 'CapacityToActivity', 'Commodity', 'CommodityType', 'ConstructionInput', 'CostEmission', 'CostFixed', 'CostInvest', 'CostVariable', 'Demand', 'DemandSpecificDistribution', 'EndOfLifeOutput', 'Efficiency', 'EfficiencyVariable', 'EmissionActivity', 'EmissionEmbodied', 'EmissionEndOfLife', 'ExistingCapacity', 'TechGroup', 'LoanLifetimeProcess', 'LoanRate', 'LifetimeProcess', 'LifetimeTech', 'Operator', 'LimitGrowthCapacity', 'LimitDegrowthCapacity', 'LimitGrowthNewCapacity', 'LimitDegrowthNewCapacity', 'LimitGrowthNewCapacityDelta', 'LimitDegrowthNewCapacityDelta', 'LimitStorageLevelFraction', 'LimitActivity', 'LimitActivityShare', 'LimitAnnualCapacityFactor', 'LimitCapacity', 'LimitCapacityShare', 'LimitNewCapacity', 'LimitNewCapacityShare', 'LimitResource', 'LimitSeasonalCapacityFactor', 'LimitTechInputSplit', 'LimitTechInputSplitAnnual', 'LimitTechOutput

In [315]:
dir = 'transport/inputs/'
transp = {}
for file in os.listdir(dir):
    #print(file)
    if file.endswith('.csv'):
        transp[file.split('_')[0]] = pd.read_csv(dir + file)


In [316]:
tech_to_remove = pd.read_csv('transport/Fuel_techs.csv')['tech'].to_list()
new_transp = {}
fuel = {}
transport = {}

for table, df in transp.items():
    if 'tech' in df.columns:
        fuel[table] = df[df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
        new_transp[table] = df[~df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
        print(table)
    else:
        new_transp[table] = df.copy()



ExistingCapacity
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
CapacityToActivity
LifetimeSurvivalCurve
CostVariable
Efficiency


In [317]:
comm = pd.read_csv('transport/Fuel_comm.csv')['commodity'].to_list()
comm



['T_h2_10', 'T_h2_100', 'T_elc_dc', 'T_ng', 'T_h2']

In [318]:
new_transp['Commodity'] = pd.read_csv('transport/Commodity.csv')
new_transp['CostFixed'] = pd.read_csv('dbs/CostFixed.csv')
new_transp['CostInvest'] = pd.read_csv('dbs/CostInvest.csv')
fuel['Commodity'] = pd.read_csv('transport/Commodity_fuel.csv')



In [319]:
for table, df in new_transp.items():
    for c in comm:
        if c in df.values:
            new_transp[table] = df[~df.isin([c]).any(axis=1)].copy().reset_index(drop=True)
            print(table)

print('\nfuel')
for table, df in fuel.items():
    for c in comm:
        if c in df.values:
            fuel[table] = df[df.isin([c]).any(axis=1)].copy().reset_index(drop=True)
            print(table)




fuel
LimitTechInputSplitAnnual
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
EmissionActivity
EmissionActivity
Efficiency
Efficiency
Efficiency
Efficiency
Efficiency
Commodity
Commodity
Commodity
Commodity


In [320]:
for table, df in new_transp.items():
    print(table)
    if ('data_id' in data[table].columns) and (table != 'DataSet'):
        if 'region' not in df.columns:
            df['data_id'] = 'TRPHR002'
        else:
            df['data_id'] = 'TRPHR' + df['region'].astype(str) + '002'

DataSource
ExistingCapacity
DataSet
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
Demand
CapacityToActivity
LifetimeSurvivalCurve
Commodity
CostVariable
TechGroup
Efficiency


In [321]:
new_transp['Technology']['tech'].to_list()

['T_BLND_DSL_ELC_HDV',
 'T_BLND_DSL_ELC_MDV',
 'T_BLND_ETH_GSL',
 'T_BLND_GSL_ELC_PHEV35',
 'T_BLND_GSL_ELC_PHEV50',
 'T_BLND_JTF',
 'T_BLND_RDSL_DSL',
 'T_BLND_SPK',
 'T_H2_HDV_REFUEL',
 'T_H2_LDV_REFUEL',
 'T_H2_MDV_REFUEL',
 'T_HDV_AJF_JFL',
 'T_HDV_AJF_SPK',
 'T_HDV_AJP_JFL',
 'T_HDV_AJP_SPK',
 'T_HDV_BIC_BEV',
 'T_HDV_BIC_DSL',
 'T_HDV_BIC_DSL_HEV',
 'T_HDV_BIC_FCEV',
 'T_HDV_BIC_GSL',
 'T_HDV_BS_BEV',
 'T_HDV_BS_CNG',
 'T_HDV_BS_DSL',
 'T_HDV_BS_DSL_HEV',
 'T_HDV_BS_DSL_PHEV',
 'T_HDV_BS_FCEV',
 'T_HDV_BS_FCHEV',
 'T_HDV_BS_GSL',
 'T_HDV_BT_BEV',
 'T_HDV_BT_CNG',
 'T_HDV_BT_DSL',
 'T_HDV_BT_DSL_HEV',
 'T_HDV_BT_DSL_PHEV',
 'T_HDV_BT_FCEV',
 'T_HDV_BT_FCHEV',
 'T_HDV_BT_GSL',
 'T_HDV_CHRG',
 'T_HDV_RF_DSL',
 'T_HDV_RF_H2',
 'T_HDV_RF_LNG',
 'T_HDV_RICP_DSL',
 'T_HDV_RICP_H2',
 'T_HDV_T_BEV',
 'T_HDV_T_DSL',
 'T_HDV_T_DSL_HEV',
 'T_HDV_T_DSL_PHEV',
 'T_HDV_T_FCEV',
 'T_HDV_T_FCHEV',
 'T_HDV_WTF_HFO',
 'T_HDV_WTF_LNG',
 'T_HDV_WTF_MDO',
 'T_LDV_BEV_CHRG',
 'T_LDV_C_BEV150',
 'T_LDV_

In [322]:
for table, df in new_transp.items():
    if "tech" in df.columns:
        print(table)
        print('')
        sub_df = df.loc[~df['tech'].isin(techs_trans)]
        if len(sub_df) > 0:
            print(sub_df['tech'].unique())
        

ExistingCapacity

CapacityFactorTech

LimitTechInputSplitAnnual

LimitAnnualCapacityFactor

CostFixed

TechGroupMember

Technology

EmissionActivity

LifetimeTech

['T_HDV_AJF_JFL_EX' 'T_HDV_AJF_JFL_N' 'T_HDV_AJF_SPK_N' 'T_HDV_AJP_JFL_EX'
 'T_HDV_AJP_JFL_N' 'T_HDV_AJP_SPK_N' 'T_HDV_BIC_BEV_N' 'T_HDV_BIC_DSL_EX'
 'T_HDV_BIC_DSL_HEV_N' 'T_HDV_BIC_DSL_N' 'T_HDV_BIC_FCEV_N'
 'T_HDV_BIC_GSL_EX' 'T_HDV_BIC_GSL_N' 'T_HDV_BS_BEV_N' 'T_HDV_BS_CNG_EX'
 'T_HDV_BS_CNG_N' 'T_HDV_BS_DSL_EX' 'T_HDV_BS_DSL_HEV_N' 'T_HDV_BS_DSL_N'
 'T_HDV_BS_DSL_PHEV_N' 'T_HDV_BS_FCEV_N' 'T_HDV_BS_FCHEV_N'
 'T_HDV_BS_GSL_EX' 'T_HDV_BS_GSL_N' 'T_HDV_BT_BEV_EX' 'T_HDV_BT_BEV_N'
 'T_HDV_BT_CNG_EX' 'T_HDV_BT_CNG_N' 'T_HDV_BT_DSL_EX' 'T_HDV_BT_DSL_HEV_N'
 'T_HDV_BT_DSL_N' 'T_HDV_BT_DSL_PHEV_N' 'T_HDV_BT_FCEV_N'
 'T_HDV_BT_FCHEV_N' 'T_HDV_BT_GSL_EX' 'T_HDV_BT_GSL_N' 'T_HDV_RF_DSL_EX'
 'T_HDV_RF_DSL_N' 'T_HDV_RF_H2_N' 'T_HDV_RF_LNG_N' 'T_HDV_RICP_DSL_EX'
 'T_HDV_RICP_DSL_N' 'T_HDV_RICP_H2_N' 'T_HDV_WTF_HFO_EX' 'T_HDV_WTF_HFO_

In [323]:
df = new_transp['LifetimeTech'].copy()


new = df.loc[df['tech'].str.endswith('_N'), 'tech'].to_list()
replacer = {tech: tech[:-2] for tech in new}

ex = df.loc[df['tech'].str.endswith('_EX'), 'tech'].to_list()
replacer.update({tech: tech[:-3] for tech in ex})
replacer

df = df.replace(replacer)

df.drop_duplicates(subset=['region', 'tech'])

new_transp['LifetimeTech'] = df.copy()

In [324]:
mgmt.update_sqlite(db_file, new_transp)


Inserting into DataSource with columns: source_id,source,notes,data_id
Inserting into ExistingCapacity with columns: region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into DataSet with columns: data_id,label,version,description,status,author,date,parent_id,changelog,notes
Inserting into CapacityFactorTech with columns: region,period,season,tod,tech,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitTechInputSplitAnnual with columns: region,period,input_comm,tech,operator,proportion,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitAnnualCapacityFactor with columns: region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CostFixed with columns: region,period,tech,vintage,cost,data_id
Inserting into TechGroupMember with columns: group_name,tech,data_id
Inserting into Technolo

In [ ]:
ind_h2 = mgmt.sqlite_to_dfs('dbs/residential/industry.sqlite')
# db_file_h2 = 'dbs/industry_h2.sqlite'
# ind_h2 = mgmt.convert_sql_to_sqlite(data_schema, db_file_h2)


{'MetaData':            element  value                         notes
 0         DB_MAJOR      3       DB major version number
 1         DB_MINOR      1       DB minor version number
 2  days_per_period    365  count of days in each period,
 'MetaDataReal':                 element  value                          notes
 0  global_discount_rate   0.03  Canadian social discount rate
 1     default_loan_rate   0.03                   Matching GDR,
 'SeasonLabel': Empty DataFrame
 Columns: [season, notes]
 Index: [],
 'SectorLabel': Empty DataFrame
 Columns: [sector, notes]
 Index: [],
 'CapacityCredit': Empty DataFrame
 Columns: [region, period, tech, vintage, credit, notes, data_source, dq_cred, dq_geog, dq_struc, dq_tech, dq_time, data_id]
 Index: [],
 'CapacityFactorProcess': Empty DataFrame
 Columns: [region, period, season, tod, tech, vintage, factor, notes, data_source, dq_cred, dq_geog, dq_struc, dq_tech, dq_time, data_id]
 Index: [],
 'CapacityFactorTech': Empty DataFrame
 Columns: 

In [153]:
techs_h2 = pd.read_csv('dbs/residential/fixers/industry_h2_techs.csv')['tech'].to_list()
comms_h2 = pd.read_csv('dbs/residential/fixers/industry_h2_comm.csv')['comm'].to_list()
tech_replace_h2 = pd.read_csv('dbs/residential/fixers/tech_replace.csv')
comm_replace_h2 = pd.read_csv('dbs/residential/fixers/comm_replace.csv')

transp.keys()

dict_keys(['DataSource', 'ExistingCapacity', 'DataSet', 'CapacityFactorTech', 'LimitTechInputSplitAnnual', 'LimitAnnualCapacityFactor', 'CostFixed', 'TechGroupMember', 'Technology', 'EmissionActivity', 'LifetimeTech', 'CostInvest', 'Demand', 'CapacityToActivity', 'LifetimeSurvivalCurve', 'Commodity', 'CostVariable', 'Efficiency'])

In [175]:
industry = {}

for table, df in transp.items():
    print(table)
    if 'tech' in df.columns:
        industry[table] = df[df['tech'].isin(techs_h2)].copy().reset_index(drop=True)
    else: 
        industry[table] = df.copy()
    
for table, df in transp.items():
    # 1. Create a boolean mask for the entire dataframe
    # 2. .any(axis=1) checks if 'True' appears in any column for that row
    mask = df.isin(comms_h2).any(axis=1)
    
    # Only add to the dictionary if we actually found matches
    if mask.any():
        industry[table] = df[mask].copy().reset_index(drop=True)

industry.keys()

DataSource
ExistingCapacity
DataSet
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
Demand
CapacityToActivity
LifetimeSurvivalCurve
Commodity
CostVariable
Efficiency


dict_keys(['DataSource', 'ExistingCapacity', 'DataSet', 'CapacityFactorTech', 'LimitTechInputSplitAnnual', 'LimitAnnualCapacityFactor', 'CostFixed', 'TechGroupMember', 'Technology', 'EmissionActivity', 'LifetimeTech', 'CostInvest', 'Demand', 'CapacityToActivity', 'LifetimeSurvivalCurve', 'Commodity', 'CostVariable', 'Efficiency'])

In [176]:
industry['Commodity']

,name,flag,description,data_id
0,T_ng,a,Natural gas for the transportation sector,FUELHR001
1,T_elc_dc,p,Electrolysis needs DC electricity,TRPHR001
2,T_h2_10,a,Gaseous H2 @ 10 bar,TRPHR001
3,T_h2_100,a,Gaseous H2 @ 100 bar,TRPHR001


In [208]:


tech_replace_h2 
comm_replace_h2 

for i in range(len(tech_replace_h2)):
    old_tech = tech_replace_h2.loc[i, 'original']
    new_tech = tech_replace_h2.loc[i, 'replace']
    for table, df in industry.items():
        if old_tech in df.values:
            industry[table] = df.replace(old_tech, new_tech)

for i in range(len(comm_replace_h2)):
    old_comm = comm_replace_h2.loc[i, 'original']
    new_comm = comm_replace_h2.loc[i, 'replace']
    for table, df in industry.items():
        if old_comm in df.values:
            industry[table] = df.replace(old_comm, new_comm)

industry['Commodity']
industry['Technology']['sector'] = 'industry'
industry['Technology']['data_id'] = 'INDHR001'
industry['Technology']
source = set()
for table, df in industry.items():
    try:
        print(table,df['data_source'].unique())
        source.update(df['data_source'].unique())
    except KeyError:
        pass
industry['DataSource'] = pd.read_csv('dbs/residential/fixers/DataSource_industry.csv')
industry

ExistingCapacity []
CapacityFactorTech []
LimitTechInputSplitAnnual ['T24']
LimitAnnualCapacityFactor [nan]
CostFixed ['T24' 'T23']
EmissionActivity ['F4' 'F5' 'T12']
LifetimeTech ['T03' 'T06']
CostInvest ['T24' 'T23']
Demand ['T17']
CapacityToActivity [nan]
CostVariable ['T23']
Efficiency [nan 'T12' 'T24' 'T06']


{'DataSource':   source_id                                             source  \
 0        F4  Government of Canada, Emission factors and ref...   
 1        F5                                           IPCC AR6   
 2       T03  DeCarolis, J. F., Jaramillo, P., Johnson, J. X...   
 3       T06  IEA. (2024). Global Hydrogen Review 2024  Assu...   
 4       T12  Dept. for Business, Energy & Industrial Strate...   
 5       T17  Natural Resources Canada. (2023b). Transportat...   
 6       T23  Plazas Nino, F., Yeganyan, R., Cannone, C., Ho...   
 7       T24  DeCarolis, J. F., Jaramillo, P., Johnson, J. X...   
 
                                                notes    data_id  
 0  The appropriate emission factors for sector an...  FUELHR001  
 1  Used for the GWP100 values for methane, carbon...  FUELHR001  
 2                                                NaN   TRPHR001  
 3                                                NaN   TRPHR001  
 4                                            

In [199]:
industry.keys()

dict_keys(['DataSource', 'ExistingCapacity', 'DataSet', 'CapacityFactorTech', 'LimitTechInputSplitAnnual', 'LimitAnnualCapacityFactor', 'CostFixed', 'TechGroupMember', 'Technology', 'EmissionActivity', 'LifetimeTech', 'CostInvest', 'Demand', 'CapacityToActivity', 'LifetimeSurvivalCurve', 'Commodity', 'CostVariable', 'Efficiency'])

In [212]:
db_industry = 'dbs/residential/industry.sqlite'
# os.remove(db_industry) if os.path.exists(db_industry) else None
# mgmt.convert_sql_to_sqlite(data_schema, db_industry)
mgmt.update_sqlite(db_industry, industry)


Inserting into DataSource with columns: source_id,source,notes,data_id
Inserting into ExistingCapacity with columns: region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into DataSet with columns: data_id,label,version,description,status,author,date,parent_id,changelog,notes
Inserting into CapacityFactorTech with columns: region,period,season,tod,tech,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitTechInputSplitAnnual with columns: region,period,input_comm,tech,operator,proportion,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitAnnualCapacityFactor with columns: region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CostFixed with columns: region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into TechGroupMem

In [ ]:
db_fuel = 'dbs/canoe_fuel.sqlite'
os.remove(db_fuel) if os.path.exists(db_fuel) else None
mgmt.convert_sql_to_sqlite(data_schema, db_fuel)
data_fuel = mgmt.sqlite_to_dfs(db_fuel)


In [189]:
# _db = mgmt.sqlite_to_dfs('dbs/canoe_all.sqlite')
mgmt.sqlite_to_excel('dbs/canoe_all.sqlite', 'canoe_all.xlsx', sheets='all')

Exporting table: CapacityCredit
Exporting table: CapacityFactorProcess
Exporting table: CapacityFactorTech
Exporting table: CapacityToActivity
Exporting table: Commodity
Exporting table: CommodityType
Exporting table: ConstructionInput
Exporting table: CostEmission
Exporting table: CostFixed
Exporting table: CostInvest
Exporting table: CostVariable
Exporting table: DataQualityCredibility
Exporting table: DataQualityGeography
Exporting table: DataQualityStructure
Exporting table: DataQualityTechnology
Exporting table: DataQualityTime
Exporting table: DataSet
Exporting table: DataSource
Exporting table: Demand
Exporting table: DemandSpecificDistribution
Exporting table: Efficiency
Exporting table: EfficiencyVariable
Exporting table: EmissionActivity
Exporting table: EmissionEmbodied
Exporting table: EmissionEndOfLife
Exporting table: EndOfLifeOutput
Exporting table: ExistingCapacity
Exporting table: LifetimeProcess
Exporting table: LifetimeSurvivalCurve
Exporting table: LifetimeTech
Expo

In [58]:
_db['Technology'].to_csv('fuel/inputs/Technology.csv', index=False)

In [59]:
_db

{'MetaData':            element  value                         notes
 0         DB_MAJOR      3       DB major version number
 1         DB_MINOR      1       DB minor version number
 2  days_per_period    363  count of days in each period,
 'MetaDataReal':                 element  value                          notes
 0  global_discount_rate   0.03  Canadian social discount rate
 1     default_loan_rate   0.03                   Matching GDR,
 'SeasonLabel':   season notes
 0   D119  None
 1   D133  None
 2   D136  None
 3   D183  None
 4   D273  None
 5   D320  None
 6   D321  None
 7   D326  None,
 'SectorLabel': Empty DataFrame
 Columns: [sector, notes]
 Index: [],
 'CapacityCredit':      region  period            tech  vintage    credit  \
 0        AB    2025     E_BIO_M-EXS     1995  0.888424   
 1        AB    2030     E_BIO_M-EXS     1995  0.888424   
 2        AB    2035     E_BIO_M-EXS     1995  0.888424   
 3        AB    2025     E_BIO_M-EXS     2000  0.888424   
 4        

In [72]:
_fuel = mgmt.sqlite_to_dfs('dbs/residential/fuel.sqlite')
_elec = mgmt.sqlite_to_dfs('dbs/residential/electricity.sqlite')

In [69]:
_fuel['Technology']['sector'] = 'fuel'
_fuel['Technology']

,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,F_E_BIO_G,p,fuel,None,None,1,1,0,0,0,0,0,0,Gaseous bioenergy distribution from fuel secto...,FUELHR002
1,F_E_BIO_M,p,fuel,None,None,1,1,0,0,0,0,0,0,Solid bioenergy distribution from fuel sector ...,FUELHR002
2,F_E_NG,p,fuel,None,None,1,1,0,0,0,0,0,0,Natural gas distribution from fuel sector to e...,FUELHR002
3,F_E_COAL,p,fuel,None,None,1,1,0,0,0,0,0,0,Coal distribution from fuel sector to electric...,FUELHR002
4,F_E_OIL,p,fuel,None,None,1,1,0,0,0,0,0,0,Oil distribution from fuel sector to electric ...,FUELHR002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,F_IMP_RDSL,p,fuel,None,None,1,1,0,0,0,0,0,0,Renewable diesel import into fuel sector,FUELHR002
75,F_IMP_SPK,p,fuel,None,None,1,1,0,0,0,0,0,0,Synthetic jet fuel import into fuel sector,FUELHR002
76,F_IMP_U_ENR,p,fuel,None,None,1,1,0,0,0,0,0,0,Enriched uranium import into fuel sector,FUELHR002
77,F_IMP_U_NAT,p,fuel,None,None,1,1,0,0,0,0,0,0,Natural uranium import into fuel sector,FUELHR002


In [73]:
for tech in fuel['Technology']['tech'].to_list():
    if tech not in _fuel['Technology']['tech'].to_list():
        print('not in fuel: ', tech)
    if tech not in _elec['Technology']['tech'].to_list():
        print('not in elec: ', tech)

not in elec:  E_T_ELC
not in elec:  F_T_H2
not in elec:  F_T_NG
not in elec:  F_T_HFO
not in elec:  F_T_GSL
not in elec:  F_T_ETH
not in elec:  F_T_DSL
not in elec:  F_T_RDSL
not in elec:  F_T_CNG
not in elec:  F_T_JTF
not in elec:  F_T_SPK
not in elec:  F_T_MDO
not in elec:  F_T_LNG
not in fuel:  T_ELC_AC_DC
not in elec:  T_ELC_AC_DC
not in fuel:  T_H2_COMP_10_100
not in elec:  T_H2_COMP_10_100
not in fuel:  T_H2_COMP_100_700
not in elec:  T_H2_COMP_100_700
not in fuel:  T_H2_distribution
not in elec:  T_H2_distribution
not in fuel:  T_I_H2_ELC_ALK
not in elec:  T_I_H2_ELC_ALK
not in fuel:  T_I_H2_ELC_PEM
not in elec:  T_I_H2_ELC_PEM
not in fuel:  T_I_H2_SMR
not in elec:  T_I_H2_SMR
not in fuel:  T_I_H2_SMR_CCS
not in elec:  T_I_H2_SMR_CCS


In [75]:
fuel['Technology'].to_csv('dbs/replaced_techs.csv', index=False)

In [258]:
techs_trans = new_transp['Technology']['tech'].to_list()
techs_trans

['T_BLND_DSL_ELC_HDV',
 'T_BLND_DSL_ELC_MDV',
 'T_BLND_ETH_GSL',
 'T_BLND_GSL_ELC_PHEV35',
 'T_BLND_GSL_ELC_PHEV50',
 'T_BLND_JTF',
 'T_BLND_RDSL_DSL',
 'T_BLND_SPK',
 'T_H2_HDV_REFUEL',
 'T_H2_LDV_REFUEL',
 'T_H2_MDV_REFUEL',
 'T_HDV_AJF_JFL',
 'T_HDV_AJF_SPK',
 'T_HDV_AJP_JFL',
 'T_HDV_AJP_SPK',
 'T_HDV_BIC_BEV',
 'T_HDV_BIC_DSL',
 'T_HDV_BIC_DSL_HEV',
 'T_HDV_BIC_FCEV',
 'T_HDV_BIC_GSL',
 'T_HDV_BS_BEV',
 'T_HDV_BS_CNG',
 'T_HDV_BS_DSL',
 'T_HDV_BS_DSL_HEV',
 'T_HDV_BS_DSL_PHEV',
 'T_HDV_BS_FCEV',
 'T_HDV_BS_FCHEV',
 'T_HDV_BS_GSL',
 'T_HDV_BT_BEV',
 'T_HDV_BT_CNG',
 'T_HDV_BT_DSL',
 'T_HDV_BT_DSL_HEV',
 'T_HDV_BT_DSL_PHEV',
 'T_HDV_BT_FCEV',
 'T_HDV_BT_FCHEV',
 'T_HDV_BT_GSL',
 'T_HDV_CHRG',
 'T_HDV_RF_DSL',
 'T_HDV_RF_H2',
 'T_HDV_RF_LNG',
 'T_HDV_RICP_DSL',
 'T_HDV_RICP_H2',
 'T_HDV_T_BEV',
 'T_HDV_T_DSL',
 'T_HDV_T_DSL_HEV',
 'T_HDV_T_DSL_PHEV',
 'T_HDV_T_FCEV',
 'T_HDV_T_FCHEV',
 'T_HDV_WTF_HFO',
 'T_HDV_WTF_LNG',
 'T_HDV_WTF_MDO',
 'T_LDV_BEV_CHRG',
 'T_LDV_C_BEV150',
 'T_LDV_

,region,tech,lifetime,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,T_H2_HDV_REFUEL,10.0,Obtained from Hydrogen Delivery Scenario Analy...,T13,NaN,NaN,NaN,NaN,NaN,TRPHRAB002
1,AB,T_H2_LDV_REFUEL,10.0,Obtained from Hydrogen Delivery Scenario Analy...,T13,NaN,NaN,NaN,NaN,NaN,TRPHRAB002
2,AB,T_H2_MDV_REFUEL,10.0,Obtained from Hydrogen Delivery Scenario Analy...,T13,NaN,NaN,NaN,NaN,NaN,TRPHRAB002
3,AB,T_HDV_AJF_JFL,24.0,Median lifetime based on Exhibits 1-3 illustra...,T02,NaN,NaN,NaN,NaN,NaN,TRPHRAB002
4,AB,T_HDV_AJF_JF,24.0,Median lifetime based on Exhibits 1-3 illustra...,T02,NaN,NaN,NaN,NaN,NaN,TRPHRAB002
...,...,...,...,...,...,...,...,...,...,...,...
1041,SK,T_MDV_T_BEV,42.0,(y) age when vehicles have 0% survival rate,NaN,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
1042,SK,T_MDV_T_DSL_HEV,42.0,(y) age when vehicles have 0% survival rate,NaN,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
1043,SK,T_MDV_T_DSL_PHEV,42.0,(y) age when vehicles have 0% survival rate,NaN,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
1044,SK,T_MDV_T_FCEV,42.0,(y) age when vehicles have 0% survival rate,NaN,NaN,NaN,NaN,NaN,NaN,TRPHRSK002
